# FineForge: reproducible Colab run

This notebook runs the whole FineForge pipeline on a Colab GPU:
validate, dedupe and shard a JSONL dataset, QLoRA fine-tune a small model, benchmark
the base model against the adapter, merge the adapter, and export a GGUF file.

**Honesty note.** The benchmark numbers come only from running this notebook. The repo
does not ship measured results. `results/benchmark.md` shows `TBD` until you run the
benchmark cells yourself, and the headline is a target, not a result.

The sample data is **synthetic** (templated toy records). Replace it with your own
dataset in the shape described in `docs/dataset-format.md`.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Clone the repository

In [ ]:
REPO_URL = "https://github.com/iboondk/fineforge"
# If you re-run this cell, delete the existing `fineforge` folder first.
!git clone $REPO_URL fineforge
%cd fineforge

## 3. Install Rust and build the `fineforge` CLI

In [ ]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

In [ ]:
!cargo build --release --manifest-path cli/Cargo.toml

## 4. Data: sample or your own

In [ ]:
# Replace with your own dataset: write JSONL in the docs/dataset-format.md shape and
# point RAW at it. The sample generator writes synthetic toy records only.
RAW = "data/sample/sample.jsonl"
!python scripts/make_sample_data.py --out $RAW

## 5. Validate the dataset

In [ ]:
!mkdir -p data/out
!./cli/target/release/fineforge validate $RAW --report data/out/validate.json

## 6. Dedupe (exact and near-duplicate)

In [ ]:
!./cli/target/release/fineforge dedupe $RAW -o data/out/dedup.jsonl --near 0.85 --stats data/out/dedupe_stats.json

## 7. Shard into train and eval splits

In [ ]:
!./cli/target/release/fineforge shard data/out/dedup.jsonl --out-dir data/out --shards 2 --eval-ratio 0.1 --seed 42

## 8. Install the trainer

In [ ]:
!pip install -e 'trainer[train]'

## 9. Optional: Hugging Face login

Only needed for gated models. The token is read from a Colab secret named `HF_TOKEN`,
never typed into the notebook.

In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login

    login(token=userdata.get("HF_TOKEN"))
    print("Logged in to Hugging Face.")
except Exception as exc:  # no secret set, or not running on Colab
    print("Skipping Hugging Face login:", exc)

## 10. Train (QLoRA)

The dry run prints the plan as JSON and trains nothing.

In [ ]:
%cd trainer
!python -m fineforge_train.train --config configs/qlora.yaml --dry-run

In [ ]:
!python -m fineforge_train.train --config configs/qlora.yaml

## 11. Benchmark: base model vs adapter

In [ ]:
!mkdir -p ../results
!python -m fineforge_train.bench.run --base-model Qwen/Qwen2.5-0.5B-Instruct --adapter runs/qlora/adapter --eval-file ../data/out/eval.jsonl --out ../results/results.json --seed 42

In [ ]:
!python -m fineforge_train.bench.report --results ../results/results.json --out ../results/benchmark.md
from IPython.display import Markdown, display

with open("../results/benchmark.md", encoding="utf-8") as fh:
    display(Markdown(fh.read()))

## 12. Merge the adapter into the base model

In [ ]:
!python -m fineforge_train.merge --base Qwen/Qwen2.5-0.5B-Instruct --adapter runs/qlora/adapter --out runs/qlora/merged

## 13. Export GGUF (llama.cpp)

Clones llama.cpp, installs its conversion requirements, and builds `llama-quantize`.
The `fineforge gguf` commands wrap the llama.cpp scripts and binaries. They do not
reimplement them.

In [ ]:
%cd ..
!git clone --depth 1 https://github.com/ggml-org/llama.cpp llama.cpp
!pip install -r llama.cpp/requirements.txt
!cmake -S llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release
!cmake --build llama.cpp/build --target llama-quantize -j

In [ ]:
import os

os.environ["LLAMA_CPP_DIR"] = os.path.abspath("llama.cpp")
QUANT_BIN = os.path.abspath("llama.cpp/build/bin/llama-quantize")
MERGED = "trainer/runs/qlora/merged"

!./cli/target/release/fineforge gguf convert $MERGED --outfile data/out/model-f16.gguf --outtype f16
!./cli/target/release/fineforge gguf quantize data/out/model-f16.gguf data/out/model-Q4_K_M.gguf --type Q4_K_M --bin $QUANT_BIN

## Done

Your numbers are in `results/results.json` and `results/benchmark.md`, both written by
this run. Quote only the values from your own run, and record the seed and eval file
hash that the results JSON contains.